In [ ]:
from __future__ import annotations

# CVERadar — single-cell Google Colab edition
# Edit the CONFIGURATION section below, then run this cell.

import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "requests", "pandas", "matplotlib"
])


import json
import os
import time
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_colwidth", 180)


# ==============================================================================

# --- Timeline ---
LOOKBACK_DAYS = 7  # @param {"type":"integer", "min":1, "max":120}
CUSTOM_START_DATE = ""  # @param {"type":"date"}
CUSTOM_END_DATE = ""  # @param {"type":"date"}

# --- Search topics (comma separated, OR matching) ---
TOPICS_TEXT = "remote code execution, ransomware"  # @param {"type":"string"}
EXACT_TOPIC_MATCH = False  # @param {"type":"boolean"}

# --- Filters ---
MIN_CVSS_SCORE = 0.0  # @param {"type":"number", "min":0, "max":10, "step":0.1}
SEVERITIES_TEXT = ""  # @param {"type":"string"}
CISA_KEV_ONLY = False  # @param {"type":"boolean"}
RESULT_LIMIT = 100  # @param {"type":"integer", "min":0}

# --- Output ---
EXPORT_CSV = True  # @param {"type":"boolean"}
EXPORT_JSON = False  # @param {"type":"boolean"}
DOWNLOAD_FILES_IN_COLAB = False  # @param {"type":"boolean"}
OUTPUT_BASENAME = "recent_cves"  # @param {"type":"string"}

# Parse the comma-separated form values.
TOPICS = [item.strip() for item in TOPICS_TEXT.split(",") if item.strip()]
SEVERITIES = {item.strip().upper() for item in SEVERITIES_TEXT.split(",") if item.strip()}
NVD_API_KEY = os.getenv("NVD_API_KEY", "").strip()
if not NVD_API_KEY:
    try:
        from google.colab import userdata
        NVD_API_KEY = (userdata.get("NVD_API_KEY") or "").strip()
    except Exception:
        # Outside Colab, or the optional secret has not been configured.
        NVD_API_KEY = ""

print(f"Topics: {TOPICS or ['all CVEs']}")
print(f"Severity filter: {sorted(SEVERITIES) or ['all severities']}")
print(f"NVD API key detected: {'yes' if NVD_API_KEY else 'no'}")


# ==============================================================================

NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
UTC = timezone.utc
PAGE_SIZE = 2_000
MAX_RETRIES = 4
REQUEST_TIMEOUT_SECONDS = 45
REQUEST_DELAY_SECONDS = 0.7 if NVD_API_KEY else 6.0

session = requests.Session()
session.headers.update({
    "User-Agent": "CVERadar/2.0 (educational defensive security tool)",
    "Accept": "application/json",
})
if NVD_API_KEY:
    session.headers["apiKey"] = NVD_API_KEY


def parse_date(value: str, *, end_of_day: bool = False) -> datetime:
    """Parse an ISO date or datetime and return a UTC-aware datetime."""
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=UTC)
    parsed = parsed.astimezone(UTC)
    if end_of_day and len(value) == 10:
        parsed = parsed + timedelta(days=1) - timedelta(milliseconds=1)
    return parsed


def resolve_time_window() -> tuple[datetime, datetime]:
    if bool(CUSTOM_START_DATE) != bool(CUSTOM_END_DATE):
        raise ValueError("Provide both custom dates, or leave both blank.")

    if CUSTOM_START_DATE and CUSTOM_END_DATE:
        start = parse_date(CUSTOM_START_DATE)
        end = parse_date(CUSTOM_END_DATE, end_of_day=True)
    else:
        if not 1 <= LOOKBACK_DAYS <= 120:
            raise ValueError("LOOKBACK_DAYS must be between 1 and 120.")
        end = datetime.now(UTC)
        start = end - timedelta(days=LOOKBACK_DAYS)

    if start >= end:
        raise ValueError("The start date must be earlier than the end date.")
    if end - start > timedelta(days=120):
        raise ValueError("NVD date-range requests cannot exceed 120 days.")
    return start, end


def nvd_timestamp(value: datetime) -> str:
    return value.astimezone(UTC).isoformat(timespec="milliseconds")


def request_json(params: dict[str, Any]) -> dict[str, Any]:
    """Request one NVD page, retrying temporary failures with backoff."""
    retryable_statuses = {403, 429, 500, 502, 503, 504}
    last_error: Exception | None = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                NVD_API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code in retryable_statuses:
                wait = float(response.headers.get("Retry-After", 2 ** attempt))
                raise requests.HTTPError(
                    f"NVD returned HTTP {response.status_code}; retrying in {wait:.1f}s",
                    response=response,
                )
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as error:
            last_error = error
            if attempt == MAX_RETRIES:
                break
            response = getattr(error, "response", None)
            retry_after = response.headers.get("Retry-After") if response is not None else None
            wait = float(retry_after) if retry_after else min(2 ** attempt, 30)
            print(f"Request attempt {attempt} failed: {error}")
            print(f"Waiting {wait:.1f} seconds before retrying...")
            time.sleep(wait)

    raise RuntimeError(f"NVD request failed after {MAX_RETRIES} attempts: {last_error}")


def fetch_cves(start: datetime, end: datetime) -> tuple[list[dict[str, Any]], dict[str, set[str]]]:
    """Retrieve all pages for every topic and de-duplicate by CVE ID."""
    records_by_id: dict[str, dict[str, Any]] = {}
    matched_topics: dict[str, set[str]] = defaultdict(set)
    searches: list[str | None] = TOPICS or [None]

    for topic_number, topic in enumerate(searches, start=1):
        params: dict[str, Any] = {
            "pubStartDate": nvd_timestamp(start),
            "pubEndDate": nvd_timestamp(end),
            "resultsPerPage": PAGE_SIZE,
            "startIndex": 0,
        }
        if topic:
            params["keywordSearch"] = topic
            if EXACT_TOPIC_MATCH:
                params["keywordExactMatch"] = ""

        label = topic or "all CVEs"
        print(f"\nSearch {topic_number}/{len(searches)}: {label!r}")

        while True:
            data = request_json(params)
            vulnerabilities = data.get("vulnerabilities", [])
            total = int(data.get("totalResults", 0))

            for item in vulnerabilities:
                cve_id = item.get("cve", {}).get("id")
                if not cve_id:
                    continue
                records_by_id[cve_id] = item
                if topic:
                    matched_topics[cve_id].add(topic)

            next_index = int(params["startIndex"]) + len(vulnerabilities)
            print(f"  Received {min(next_index, total):,} / {total:,}")
            if not vulnerabilities or next_index >= total:
                break

            params["startIndex"] = next_index
            time.sleep(REQUEST_DELAY_SECONDS)

        if topic_number < len(searches):
            time.sleep(REQUEST_DELAY_SECONDS)

    return list(records_by_id.values()), matched_topics


# ==============================================================================

CVSS_VERSIONS = [
    ("cvssMetricV40", "4.0"),
    ("cvssMetricV31", "3.1"),
    ("cvssMetricV30", "3.0"),
    ("cvssMetricV2", "2.0"),
]


def english_text(items: list[dict[str, Any]], default: str = "") -> str:
    return next(
        (item.get("value", "") for item in items if item.get("lang") == "en"),
        default,
    )


def extract_cvss(cve: dict[str, Any]) -> dict[str, Any]:
    metrics = cve.get("metrics", {})
    for key, version in CVSS_VERSIONS:
        entries = metrics.get(key, [])
        if not entries:
            continue
        metric = next((entry for entry in entries if entry.get("type") == "Primary"), entries[0])
        data = metric.get("cvssData", {})
        return {
            "CVSS Version": version,
            "CVSS Score": data.get("baseScore"),
            "Severity": data.get("baseSeverity") or metric.get("baseSeverity") or "UNKNOWN",
            "CVSS Vector": data.get("vectorString", ""),
            "CVSS Source": metric.get("source", ""),
        }
    return {
        "CVSS Version": "",
        "CVSS Score": None,
        "Severity": "UNKNOWN",
        "CVSS Vector": "",
        "CVSS Source": "",
    }


def extract_cwes(cve: dict[str, Any]) -> str:
    values: list[str] = []
    for weakness in cve.get("weaknesses", []):
        text = english_text(weakness.get("description", []))
        if text and text not in values:
            values.append(text)
    return ", ".join(values)


def make_dataframe(
    raw_items: list[dict[str, Any]],
    matched_topics: dict[str, set[str]],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for item in raw_items:
        cve = item.get("cve", {})
        cve_id = cve.get("id", "")
        references = [ref.get("url", "") for ref in cve.get("references", []) if ref.get("url")]
        row = {
            "CVE ID": cve_id,
            "Published": cve.get("published"),
            "Last Modified": cve.get("lastModified"),
            "Status": cve.get("vulnStatus", ""),
            "Description": english_text(cve.get("descriptions", []), "No English description available"),
            "CWE": extract_cwes(cve),
            "Matched Topics": ", ".join(sorted(matched_topics.get(cve_id, set()))),
            "CISA KEV": bool(cve.get("cisaExploitAdd")),
            "CISA KEV Added": cve.get("cisaExploitAdd", ""),
            "CISA Due Date": cve.get("cisaActionDue", ""),
            "Required Action": cve.get("cisaRequiredAction", ""),
            "NVD URL": f"https://nvd.nist.gov/vuln/detail/{cve_id}",
            "References": " | ".join(references),
        }
        row.update(extract_cvss(cve))
        rows.append(row)

    columns = [
        "CVE ID", "Published", "Last Modified", "Severity", "CVSS Score",
        "CVSS Version", "CVSS Vector", "CVSS Source", "Status", "CISA KEV",
        "CISA KEV Added", "CISA Due Date", "Matched Topics", "CWE", "Description",
        "Required Action", "NVD URL", "References",
    ]
    frame = pd.DataFrame(rows, columns=columns)
    if frame.empty:
        return frame

    frame["Published"] = pd.to_datetime(frame["Published"], utc=True, errors="coerce")
    frame["Last Modified"] = pd.to_datetime(frame["Last Modified"], utc=True, errors="coerce")
    frame["CVSS Score"] = pd.to_numeric(frame["CVSS Score"], errors="coerce")
    frame["Severity"] = frame["Severity"].fillna("UNKNOWN").str.upper()

    if MIN_CVSS_SCORE > 0:
        frame = frame[frame["CVSS Score"].fillna(-1) >= MIN_CVSS_SCORE]
    if SEVERITIES:
        frame = frame[frame["Severity"].isin(SEVERITIES)]
    if CISA_KEV_ONLY:
        frame = frame[frame["CISA KEV"]]

    frame = frame.sort_values(["Published", "CVE ID"], ascending=[False, False])
    if RESULT_LIMIT > 0:
        frame = frame.head(RESULT_LIMIT)
    return frame.reset_index(drop=True)


# ==============================================================================

start_time, end_time = resolve_time_window()
print(f"Request window (UTC): {start_time.isoformat()} to {end_time.isoformat()}")

raw_cves, topic_matches = fetch_cves(start_time, end_time)
df = make_dataframe(raw_cves, topic_matches)

print("\n" + "=" * 72)
print(f"Unique CVEs retrieved before filters: {len(raw_cves):,}")
print(f"Rows retained after filters and limit: {len(df):,}")
print("=" * 72)

if df.empty:
    print("No CVEs matched the selected timeline, topics, and filters.")
else:
    display_columns = [
        "CVE ID", "Published", "Severity", "CVSS Score",
        "CISA KEV", "Matched Topics", "Description", "NVD URL",
    ]
    display(df[display_columns])


# ==============================================================================

if df.empty:
    print("Nothing to chart.")
else:
    severity_order = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "UNKNOWN"]
    severity_counts = df["Severity"].value_counts().reindex(severity_order, fill_value=0)
    daily_counts = df.set_index("Published").resample("D").size()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    severity_counts.plot(kind="bar", ax=axes[0], color=["#7f1d1d", "#dc2626", "#f59e0b", "#3b82f6", "#6b7280"])
    axes[0].set_title("CVEs by severity")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=30)

    daily_counts.plot(kind="line", marker="o", ax=axes[1], color="#2563eb")
    axes[1].set_title("Published CVEs over time")
    axes[1].set_xlabel("Publication day (UTC)")
    axes[1].set_ylabel("Count")
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


# ==============================================================================

exported_files: list[Path] = []
safe_basename = "".join(char for char in OUTPUT_BASENAME if char.isalnum() or char in {"-", "_"}).strip("_")
if not safe_basename:
    safe_basename = "recent_cves"
timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")

if df.empty:
    print("No export was created because the result set is empty.")
else:
    if EXPORT_CSV:
        csv_path = Path(f"{safe_basename}_{timestamp}.csv")
        df.to_csv(csv_path, index=False)
        exported_files.append(csv_path)

    if EXPORT_JSON:
        json_path = Path(f"{safe_basename}_{timestamp}.json")
        df.to_json(json_path, orient="records", indent=2, date_format="iso")
        exported_files.append(json_path)

    for path in exported_files:
        print(f"Saved {path.resolve()}")

    if DOWNLOAD_FILES_IN_COLAB and exported_files:
        try:
            from google.colab import files
            for path in exported_files:
                files.download(str(path))
        except ImportError:
            print("Automatic download is available only in Google Colab.")

